# Support Vector Machine (SVM) Assignment

## Predicting High and Low Sales

For this assignment, you will create a new target variable called SalesLevel.

- Low = Sales below the median

- High = Sales at or above the median

In [2]:
# Ananth Kashibhatla
# 13 October 2026
# Professor Dawn Rywalt

# Import the necessary libraries
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt   

## We will use the Advertising dataset to test our SVM capability, import the dataset and EDA first

In [3]:
# Load the Advertising dataset file into the dataframe variable initialized
df_advertising_dataset = pd.read_csv("Advertising.csv")

# Explore the dataset structures
print(df_advertising_dataset.head())
print(df_advertising_dataset.tail())
print(len(df_advertising_dataset))
print(df_advertising_dataset.shape)
print(df_advertising_dataset.dtypes)


   Unnamed: 0     TV  Radio  Newspaper  Sales  High_Sales
0           1  230.1   37.8       69.2   22.1           1
1           2   44.5   39.3       45.1   10.4           0
2           3   17.2   45.9       69.3    9.3           0
3           4  151.5   41.3       58.5   18.5           1
4           5  180.8   10.8       58.4   12.9           1
     Unnamed: 0     TV  Radio  Newspaper  Sales  High_Sales
195         196   38.2    3.7       13.8    7.6           0
196         197   94.2    4.9        8.1    9.7           0
197         198  177.0    9.3        6.4   12.8           0
198         199  283.6   42.0       66.2   25.5           1
199         200  232.1    8.6        8.7   13.4           1
200
(200, 6)
Unnamed: 0      int64
TV            float64
Radio         float64
Newspaper     float64
Sales         float64
High_Sales      int64
dtype: object


## Create the Target Variable

Create a new column called SalesLevel.

Use the median sales value to classify each observation:

- If Sales is below the median → Low

- If Sales is greater than or equal to the median → High

In [4]:
median_sales = df_advertising_dataset["Sales"].median()

# Create a new column called SalesLevel and classify it as high
df_advertising_dataset["SalesLevel"] = "High"

# Classify any sales below the median as Low 
df_advertising_dataset.loc[
    df_advertising_dataset["Sales"] < median_sales, 
    "SalesLevel"
] = "Low"

# Display the median 
print("Median Sales: ", median_sales)

# use the .values method to find how many observations are classified as High or Low
print(df_advertising_dataset["SalesLevel"].value_counts())



Median Sales:  12.9
SalesLevel
High    102
Low      98
Name: count, dtype: int64


### Questions

1. How many observations are classified as High?

    102 observations are classified as High.

2. How many observations are classified as Low?

    98 observations are classified as Low. 

3. Why is it important to define the target variable before training a classification model?

    Defining a target variable before training a classification model is important so that we actually know we are trying to predict. Especially with out SVM models, the SalesLevel column classifies by High or Low, we need a column that can be used by our SVM to learn and predict the patterns we are looking for. 


## Split the Data

For this assignment, use the advertising expenditures as your features:

- TV

- Radio

- Newspaper

The target variable is:

- SalesLevel

Split the data into training and testing sets.

Use:

- 80% for training

- 20% for testing

- random_state=42

- stratify=y

In [5]:
# Split the data into features or TV, Radio, Newspaper
X = df_advertising_dataset[["TV", "Radio", "Newspaper"]]

# Split the data into the target variable or SalesLevel
y = df_advertising_dataset['SalesLevel']

# Perform the training and test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42, stratify=y)

# Display the training and testing data
print("Training data: ", X_train)
print("Testing data: ", X_test)

Training data:          TV  Radio  Newspaper
169  284.3   10.6        6.4
122  224.0    2.4       15.6
37    74.7   49.4       45.7
190   39.5   41.1        5.8
155    4.1   11.6        5.7
..     ...    ...        ...
28   248.8   27.1       22.9
144   96.2   14.8       38.9
164  117.2   14.7        5.4
6     57.5   32.8       23.5
130    0.7   39.6        8.7

[160 rows x 3 columns]
Testing data:          TV  Radio  Newspaper
67   139.3   14.5       10.2
86    76.3   27.5       16.0
191   75.5   10.8        6.0
192   17.2    4.1       31.6
45   175.1   22.5       31.5
8      8.6    2.1        1.0
55   198.9   49.4       60.0
170   50.0   11.6       18.4
132    8.4   27.2        2.1
162  188.4   18.1       25.6
12    23.8   35.1       65.9
195   38.2    3.7       13.8
3    151.5   41.3       58.5
60    53.5    2.0       21.4
183  287.6   43.0       71.8
30   292.9   28.3       43.2
149   44.7   25.8       20.6
97   184.9   21.0       22.0
163  163.5   36.8        7.4
76    27.5    1.6

### Question:  Why should we use stratify=y when splitting this classification dataset?

    We should use stratify=y when splitting this classification dataset because it controls how the data is divided so that the training and testing sets maintain approximately the same proportion of target classes as the original dataset.

## Standardize the Features

SVM models are sensitive to the scale of the features.

Use StandardScaler to standardize the feature data.

Create:

- `X_train_scaled`

- `X_test_scaled`

In [6]:
# Create the StandardScaler
scaler = StandardScaler()

# Let's fit the training data into X_train_scaled
X_train_scaled = scaler.fit_transform(X_train)

# Let's fit the testing data into X_test_scaled
X_test_scaled = scaler.transform(X_test)

### Questions

1. Why does an SVM benefit from standardized data?

    SVM benefits from standarized data because it relies heavily on geometric distances(such as creating that hyperplane) and having that standardized data prevents features with large numeric ranges from closing that gap. 

2. Why should the scaler be fit using only the training data?

    Scalar should be fit using only the training data because we want to prevent data leakage on our training dataset; we shouldn't use scalar on testing data because that needs to be remain unseen for our model to make accurate predictions. 


## Build the SVM Model

Import SVC from sklearn.svm.

Create an SVM classifier using:

`SVC(kernel='linear')`

Train the model using the training data.

Make predictions using the testing data.

Display the predictions.

In [7]:
# Create an SVM classifier using SVC(kernel='linear')
svm_linear = SVC(kernel="linear")

# Train the model using the training data 
svm_linear.fit(X_train_scaled, y_train)

# Make predictions using the testing data
y_pred = svm_linear.predict(X_test_scaled)

# Display the predictions
print("Predictions: ", y_pred)

Predictions:  ['Low' 'Low' 'Low' 'Low' 'High' 'Low' 'High' 'Low' 'Low' 'High' 'Low'
 'Low' 'High' 'Low' 'High' 'High' 'Low' 'High' 'High' 'Low' 'High' 'Low'
 'Low' 'Low' 'Low' 'Low' 'Low' 'High' 'Low' 'High' 'Low' 'High' 'High'
 'High' 'High' 'Low' 'High' 'High' 'Low' 'High']


## Evaluate the Model

Calculate the following:

- Accuracy

- Precision

- Recall

- Confusion matrix


In [8]:
# Accuracy 
accuracy = accuracy_score(y_test, y_pred)

# Precision 
precision = precision_score(y_test, y_pred, pos_label = "High")

# Recall 
recall = recall_score(y_test, y_pred, pos_label = "High")

# Confusion Matrix 
cm = confusion_matrix(y_test, y_pred, labels=["Low", "High"])

# Display the metrics 
print("Accuracy:", accuracy)
print("Accuracy(%):", accuracy * 100)
print("Precision:", precision)
print("Recall:", recall)
print("Confusion Matrix:", cm)


Accuracy: 0.95
Accuracy(%): 95.0
Precision: 1.0
Recall: 0.9
Confusion Matrix: [[20  0]
 [ 2 18]]


### Questions

1. What percentage of observations were classified correctly?

    The Linear SVM achieved 95% accuracy. 

2. What is the precision for the High class?

    The precision for the High class is 1.0, meaning 100% of our predictions were High. 

3. What is the recall for the High class?

    The recall for the High class is 0.9, meaning 90% of our dataset was correctly predicted as the High sales.

4. How many High-sales observations were correctly predicted?

    18 High-sales observations were correctly predicted. 

5. How many Low-sales observations were incorrectly classified as High?

    No Low-sales observations were incorrectly classified as High.



## Try Different Kernels

Create two additional SVM models.

### Model 1 – Linear

`SVC(kernel='linear')`

### Model 2 – RBF

`SVC(kernel='rbf')`

Train and evaluate both models for accuracy, precision and recall.

In [9]:
# Model 1 - Linear 
model_linear = SVC(kernel="linear")

model_linear.fit(X_train_scaled, y_train)

linear_prediction = model_linear.predict(X_test_scaled)

linear_accuracy = accuracy_score(
    y_test, 
    linear_prediction
)

linear_precision = precision_score(
    y_test, linear_prediction, pos_label="High"
)

linear_recall = recall_score(
    y_test, linear_prediction, pos_label="High"
)

print("Linear Accuracy:", linear_accuracy)
print("Linear Precision:", linear_precision)
print("Linear Recall:", linear_recall)

Linear Accuracy: 0.95
Linear Precision: 1.0
Linear Recall: 0.9


In [10]:
# Model 2 : RBF Kernel 

model_rbf = SVC(kernel="rbf")

model_rbf.fit(X_train_scaled, y_train)

rbf_predict = model_rbf.predict(X_test_scaled)

rbf_accuracy = accuracy_score(y_test, rbf_predict)

rbf_precision = precision_score(y_test, rbf_predict, pos_label = "High")

rbf_recall = recall_score(y_test, rbf_predict,  pos_label = "High")

print("RBF Accuracy:", rbf_accuracy)
print("RBF Precision:", rbf_precision)
print("RBF Recall:", rbf_recall)



RBF Accuracy: 0.95
RBF Precision: 1.0
RBF Recall: 0.9


### Questions

1. Which kernel produced the highest accuracy?

    Both the linear kernel and RBF produced the same accuracy of 95%. 

2. Which model had the best recall?

    Both models has the best recall, with 90% for linear kernel and 90% for the RBF kernel. 

3. Why might an RBF kernel perform differently from a linear kernel?

    An RBF kernel has more nuances in terms of non-linear and curvatures with SVM capability of our models and the dataset. Whereas, the linear kernel is more the classic linear, 2-D hyperplane. For more complex and curved dataset patterns, a linear kernel will not perform as well as the RBF kernel, simply due to its capability. 

4. Which model would you choose for this dataset? Explain your answer.

    Honestly, I would choose hte linear kernel simply because we are not working with complex, nunaced datasets. We just have three features and one target at most, I don't see the need for RBF kernel, especially when both models tested to have the same accuracy and recall percentages. 


## Experiment with C

The parameter C controls how strongly the SVM penalizes classification errors.

Test the following values and evaluate all C values for accuracy.

- `C = 0.01`

- `C = 0.1`

- `C = 1`

- `C = 10`

- `C = 100`

In [11]:
# Experiment with different C values 
C_values = [0.01, 0.1, 1, 10, 100]

# Create a for loop for each C value and create a SVM model to assess model capability
for C in C_values: 

    # Create the linear SVM model 
    linear_model = SVC(kernel="linear", C=C)

    # Train the model with scaled training and y testing data
    linear_model.fit(X_train_scaled, y_train)

    # Make predictions with the testing data
    linear_predictions = linear_model.predict(X_test_scaled)

    # Find the accuracy for each SVM with each C value
    linear_accuracy = accuracy_score(y_test, linear_predictions)

    # Display all the results 
    print("C value: ", C, "Accuracy for each test C value: ", linear_accuracy)





C value:  0.01 Accuracy for each test C value:  0.9
C value:  0.1 Accuracy for each test C value:  0.9
C value:  1 Accuracy for each test C value:  0.95
C value:  10 Accuracy for each test C value:  0.925
C value:  100 Accuracy for each test C value:  0.925


### Questions

1. Which value of C produced the highest accuracy?

    The value of C = 1 produced the highest accuracy of 95%. 

2. What happens to the model as C becomes larger?

    Based on the various C values we tested, I noticed that as our C becomes larger, the SVM model capability is becoming more error prone and becoming harder to classify out training dataset observations. 

3. Why might a very large value of C potentially cause overfitting?

    A very large value of C can potentially cause overfitting because it over does the training dataset observations to the point it forces the model to classify each and every single data point. 


## Challenge Question

Suppose a new advertising campaign spends:

- TV = 200

- Radio = 40

- Newspaper = 20

Use your best-performing SVM model to predict whether the expected sales level is High or Low and Explain.

In [12]:
# Create a dataframe to hold the new advertising campaign 
df_new_campaign = pd.DataFrame({ 
    "TV": [200], 
    "Radio": [40], 
    "Newspaper": [20]
})

# Standardize via scaler
scaler_new_campaign = scaler.transform(df_new_campaign)

# Make predictions with out best performing SVM model 
new_campaign_predictions = model_linear.predict(scaler_new_campaign)

# Display the predictions
print("Predictions: ", new_campaign_predictions[0])

Predictions:  High


### Challenge Question — Written Explanation

The predictions using my best-performing SVM model is High for the Sales Level. My best performing model for this dataset is the Linear SVM, which achieved a 95% accuracy. 